# Phase 1 EDA — Combined Cross-Dataset Analysis

This notebook aggregates the per-asset EDA results and performs
cross-dataset comparability analysis (Step 1.6), produces the
EDA summary document (Step 1.8), and saves the final combined
deliverables required by Gate 2.

**Prerequisites:** Run all 4 per-asset EDA notebooks first.

**Outputs:**
- `results/eda/eda_summary.json` — combined summary for Gate 2
- `results/visualization_examples.json` — 12 fixed examples (Meta Plan)
- Cross-dataset comparison plots in `results/eda_plots/combined/`

In [1]:
# ============================================================
# Cell 1 — Setup & Load Per-Asset Reports
# ============================================================
import sys, os, json, glob
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yaml

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_ROOT = "/content/drive/MyDrive/seismic-first-break-picking" if IN_COLAB else os.getcwd()
config_path = os.path.join(REPO_ROOT, "configs", "datasets.yaml")
with open(config_path) as f:
    CFG = yaml.safe_load(f)

ASSETS = CFG["assets"]["names"]
EDA_JSON_DIR = os.path.join(REPO_ROOT, CFG["paths"]["eda_json"])
COMBINED_PLOT_DIR = os.path.join(REPO_ROOT, CFG["paths"]["eda_plots"], "combined")
os.makedirs(COMBINED_PLOT_DIR, exist_ok=True)

# Load all per-asset reports
reports = {}
for asset in ASSETS:
    rpath = os.path.join(EDA_JSON_DIR, f"{asset}_eda_report.json")
    if os.path.exists(rpath):
        with open(rpath) as f:
            reports[asset] = json.load(f)
        print(f"Loaded: {rpath}")
    else:
        print(f"WARNING: Missing report for {asset} — run 01_eda_{asset}.ipynb first!")

# Also load Phase 0.5 env report
env_report_path = os.path.join(REPO_ROOT, "results", "00_environment_report.json")
with open(env_report_path) as f:
    env_report = json.load(f)
print(f"\nLoaded: {env_report_path}")

if len(reports) < 4:
    print("\n⚠ Not all 4 per-asset reports available. Results will be partial.")
else:
    print(f"\nAll 4 per-asset EDA reports loaded successfully.")

Mounted at /content/drive
Loaded: /content/drive/MyDrive/seismic-first-break-picking/results/eda/brunswick_eda_report.json
Loaded: /content/drive/MyDrive/seismic-first-break-picking/results/eda/halfmile_eda_report.json
Loaded: /content/drive/MyDrive/seismic-first-break-picking/results/eda/lalor_eda_report.json
Loaded: /content/drive/MyDrive/seismic-first-break-picking/results/eda/sudbury_eda_report.json

Loaded: /content/drive/MyDrive/seismic-first-break-picking/results/00_environment_report.json

All 4 per-asset EDA reports loaded successfully.


In [2]:
# ============================================================
# Cell 2 — Cross-Dataset Comparability Table (Step 1.6)
# ============================================================
print("=" * 80)
print("CROSS-DATASET COMPARISON TABLE")
print("=" * 80)

header = f"{'Property':<30} {'Brunswick':>12} {'Halfmile':>12} {'Lalor':>12} {'Sudbury':>12}"
print(header)
print("-" * len(header))

rows = []
for prop_name, cfg_key in [
    ("Total traces", "total_traces"),
    ("Samples/trace", "samp_num"),
    ("Sample rate (µs)", "samp_rate_us"),
    ("Time window (ms)", "time_window_ms"),
    ("COORD_SCALE", "coord_scale"),
    ("HT_SCALE", "ht_scale"),
    ("Labeled traces", "labeled_traces"),
    ("Label fraction", "labeled_fraction"),
]:
    vals = []
    for asset in ASSETS:
        v = CFG["asset_meta"][asset][cfg_key]
        if isinstance(v, float) and v < 1:
            vals.append(f"{v*100:.1f}%")
        elif isinstance(v, int) and v > 10000:
            vals.append(f"{v:,}")
        else:
            vals.append(str(v))
    row = f"{prop_name:<30} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12} {vals[3]:>12}"
    print(row)

# EDA-derived stats
print("\n--- From EDA Notebooks ---")
for prop_name, json_path in [
    ("Unique shots", ["shot_gather_stats", "n_shots"]),
    ("Median traces/shot", ["shot_gather_stats", "gather_size", "median"]),
    ("Offset max", ["shot_gather_stats", "offset_range", "global_max"]),
    ("Dead trace %", ["amplitude_stats", "dead_fraction_estimate"]),
    ("Clipped trace %", ["amplitude_stats", "clipped_fraction_estimate"]),
    ("FB time median (ms)", ["label_audit", "fb_time_ms", "median"]),
    ("FB time std (ms)", ["label_audit", "fb_time_ms", "std"]),
    ("Mispick rate (%)", ["label_quality", "outlier_fraction"]),
]:
    vals = []
    for asset in ASSETS:
        if asset not in reports:
            vals.append("N/A")
            continue
        obj = reports[asset]
        try:
            for key in json_path:
                obj = obj[key]
            if isinstance(obj, float):
                if "fraction" in prop_name.lower() or "%" in prop_name:
                    vals.append(f"{obj*100:.3f}%")
                else:
                    vals.append(f"{obj:.1f}")
            elif isinstance(obj, int):
                vals.append(f"{obj:,}")
            else:
                vals.append(str(obj))
        except (KeyError, TypeError):
            vals.append("N/A")
    row = f"{prop_name:<30} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12} {vals[3]:>12}"
    print(row)

CROSS-DATASET COMPARISON TABLE
Property                          Brunswick     Halfmile        Lalor      Sudbury
----------------------------------------------------------------------------------
Total traces                      4,496,540    1,099,559    2,424,923    1,810,220
Samples/trace                           751          751         1501         1001
Sample rate (µs)                       2000         2000         1000         2000
Time window (ms)                       1500         1500         1500         2000
COORD_SCALE                             -10            1          -10          100
HT_SCALE                                -10       -10000          -10           10
Labeled traces                    3,733,221      993,189    1,211,857      200,338
Label fraction                        83.0%        90.3%        50.0%        11.1%

--- From EDA Notebooks ---
Unique shots                          1,541          690          907        1,016
Median traces/shot          

In [3]:
# ============================================================
# Cell 3 — Combined FB Time Distribution Overlay
# ============================================================
print("=== Combined First Break Time Distributions ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {"brunswick": "steelblue", "halfmile": "darkorange",
          "lalor": "forestgreen", "sudbury": "crimson"}

for ax, asset in zip(axes.flat, ASSETS):
    if asset not in reports:
        ax.text(0.5, 0.5, f"{asset}\nNo data", ha="center", va="center")
        continue
    fb = reports[asset].get("label_audit", {}).get("fb_time_ms", {})
    if fb:
        ax.set_title(f"{asset.capitalize()} (median={fb['median']:.0f}ms, std={fb['std']:.0f}ms)")
        # We don't have the raw values here, but we can show the summary stats
        stats = [fb['min'], fb['p5'], fb['p25'], fb['median'], fb['p75'], fb['p95'], fb['max']]
        labels = ['min', 'p5', 'p25', 'med', 'p75', 'p95', 'max']
        ax.barh(labels, stats, color=colors[asset], alpha=0.8)
        ax.set_xlabel("FB Time (ms)")

fig.suptitle("First Break Time Statistics — All Assets", fontsize=14)
fig.tight_layout()
save_path = os.path.join(COMBINED_PLOT_DIR, "combined_fb_stats.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

=== Combined First Break Time Distributions ===
Saved: /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/combined/combined_fb_stats.png


In [4]:
# ============================================================
# Cell 4 — Data Scale & Heterogeneity Analysis
# ============================================================
print("=== Critical Heterogeneity Analysis ===")
print()

# 1. Sample rate mismatch
rates = {a: CFG["asset_meta"][a]["samp_rate_us"] for a in ASSETS}
unique_rates = set(rates.values())
print(f"1. Sample rates: {rates}")
if len(unique_rates) > 1:
    print(f"   ⚠ MISMATCH: {len(unique_rates)} different rates found: {unique_rates}")
    print(f"   → Lalor uses 1000µs (1ms), others use 2000µs (2ms)")
    print(f"   → Resampling or interpolation needed before combined training.")
else:
    print(f"   ✓ All assets use the same rate: {unique_rates.pop()}µs")

# 2. Trace length mismatch
samp_nums = {a: CFG["asset_meta"][a]["samp_num"] for a in ASSETS}
unique_samps = set(samp_nums.values())
print(f"\n2. Samples per trace: {samp_nums}")
if len(unique_samps) > 1:
    print(f"   ⚠ MISMATCH: {len(unique_samps)} different lengths: {unique_samps}")
    print(f"   → Need padding, cropping, or resampling to unify.")
    # Calculate time windows
    for a in ASSETS:
        tw = (samp_nums[a] - 1) * rates[a] / 1000.0
        print(f"     {a}: {samp_nums[a]} samples × {rates[a]/1000:.1f}ms = {tw:.0f}ms window")
else:
    print(f"   ✓ All assets have the same trace length: {unique_samps.pop()}")

# 3. Coordinate scale mismatch
coord_scales = {a: CFG["asset_meta"][a]["coord_scale"] for a in ASSETS}
print(f"\n3. COORD_SCALE: {coord_scales}")
print(f"   → Coordinates are in different units across assets.")
print(f"   → After scaling: Brunswick/Lalor divide by 10, Halfmile×1, Sudbury×100")
print(f"   → Source-receiver offsets will be in different scales.")

# 4. Label coverage imbalance
print(f"\n4. Label coverage:")
for a in ASSETS:
    frac = CFG["asset_meta"][a]["labeled_fraction"]
    total = CFG["asset_meta"][a]["total_traces"]
    labeled = CFG["asset_meta"][a]["labeled_traces"]
    print(f"   {a:10s}: {labeled:>10,} / {total:>10,} = {frac*100:>6.2f}%")
print(f"   → Sudbury has ~8× fewer labeled traces than Brunswick.")
print(f"   → Need balanced sampling strategy for combined training.")

# 5. First break time ranges
print(f"\n5. First break time ranges (ms):")
for asset in ASSETS:
    if asset in reports:
        fb = reports[asset].get("label_audit", {}).get("fb_time_ms", {})
        if fb:
            print(f"   {asset:10s}: [{fb['min']:.0f} — {fb['max']:.0f}], "
                  f"P5-P95=[{fb['p5']:.0f} — {fb['p95']:.0f}]")

=== Critical Heterogeneity Analysis ===

1. Sample rates: {'brunswick': 2000, 'halfmile': 2000, 'lalor': 1000, 'sudbury': 2000}
   ⚠ MISMATCH: 2 different rates found: {2000, 1000}
   → Lalor uses 1000µs (1ms), others use 2000µs (2ms)
   → Resampling or interpolation needed before combined training.

2. Samples per trace: {'brunswick': 751, 'halfmile': 751, 'lalor': 1501, 'sudbury': 1001}
   ⚠ MISMATCH: 3 different lengths: {1001, 1501, 751}
   → Need padding, cropping, or resampling to unify.
     brunswick: 751 samples × 2.0ms = 1500ms window
     halfmile: 751 samples × 2.0ms = 1500ms window
     lalor: 1501 samples × 1.0ms = 1500ms window
     sudbury: 1001 samples × 2.0ms = 2000ms window

3. COORD_SCALE: {'brunswick': -10, 'halfmile': 1, 'lalor': -10, 'sudbury': 100}
   → Coordinates are in different units across assets.
   → After scaling: Brunswick/Lalor divide by 10, Halfmile×1, Sudbury×100
   → Source-receiver offsets will be in different scales.

4. Label coverage:
   brunswi

In [5]:
# ============================================================
# Cell 5 — Shot Gather Size Comparison
# ============================================================
print("=== Shot Gather Size Comparison ===")

fig, ax = plt.subplots(figsize=(12, 6))
positions = []
labels_list = []

for i, asset in enumerate(ASSETS):
    if asset in reports:
        gs = reports[asset].get("shot_gather_stats", {}).get("gather_size", {})
        if gs:
            stats = [gs['min'], gs['p5'], gs['median'], gs['p95'], gs['max']]
            ax.barh(i, gs['median'], color=colors[asset], alpha=0.7,
                    label=f"{asset} (n={reports[asset]['shot_gather_stats']['n_shots']:,})")
            ax.errorbar(gs['median'], i,
                       xerr=[[gs['median'] - gs['p5']], [gs['p95'] - gs['median']]],
                       fmt='none', color='black', capsize=5)
            positions.append(i)
            labels_list.append(f"{asset.capitalize()}\n(n={reports[asset]['shot_gather_stats']['n_shots']:,})")

ax.set_yticks(positions)
ax.set_yticklabels(labels_list)
ax.set_xlabel("Traces per Shot Gather (bars=median, whiskers=P5-P95)")
ax.set_title("Shot Gather Size Comparison")
fig.tight_layout()
save_path = os.path.join(COMBINED_PLOT_DIR, "combined_gather_sizes.png")
fig.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

=== Shot Gather Size Comparison ===
Saved: /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/combined/combined_gather_sizes.png


In [6]:
# ============================================================
# Cell 6 — SPARE1 vs FIRST_BREAK_TIME Summary
# ============================================================
print("=== SPARE1 vs FIRST_BREAK_TIME — All Assets ===")
for asset in ASSETS:
    if asset not in reports:
        continue
    comparison = reports[asset].get("spare1_vs_fbt")
    if comparison is None:
        print(f"  {asset}: FIRST_BREAK_TIME not present")
    elif comparison.get("n_both_labeled", 0) == 0:
        print(f"  {asset}: FIRST_BREAK_TIME present but no overlap")
    else:
        match_pct = comparison['exact_match_fraction'] * 100
        print(f"  {asset}: {comparison['n_both_labeled']:,} overlap, "
              f"{match_pct:.1f}% exact match, diff_std={comparison['diff_std']:.3f}ms")

=== SPARE1 vs FIRST_BREAK_TIME — All Assets ===
  brunswick: FIRST_BREAK_TIME present but no overlap
  halfmile: FIRST_BREAK_TIME present but no overlap
  lalor: FIRST_BREAK_TIME present but no overlap
  sudbury: FIRST_BREAK_TIME present but no overlap


In [7]:
# ============================================================
# Cell 7 — Label Quality Summary
# ============================================================
print("=== Label Quality Summary — All Assets ===")
for asset in ASSETS:
    if asset not in reports:
        continue
    lq = reports[asset].get("label_quality", {})
    if lq.get("label_quality_available"):
        print(f"  {asset:10s}: checked {lq['n_shots_checked']} shots, "
              f"{lq['total_labeled_checked']:,} labels, "
              f"outliers={lq['total_outliers_3sigma']} ({lq['outlier_fraction']*100:.3f}%), "
              f"residual_std={lq['residual_stats']['std']:.2f}ms")
    else:
        print(f"  {asset:10s}: quality check not available")

=== Label Quality Summary — All Assets ===
  brunswick : checked 200 shots, 484,018 labels, outliers=1908 (0.394%), residual_std=8.95ms
  halfmile  : checked 200 shots, 287,865 labels, outliers=1354 (0.470%), residual_std=7.62ms
  lalor     : checked 200 shots, 261,879 labels, outliers=3623 (1.383%), residual_std=8.37ms
  sudbury   : checked 200 shots, 55,085 labels, outliers=619 (1.124%), residual_std=6.79ms


In [8]:
# ============================================================
# Cell 8 — Merge Visualization Examples
# ============================================================
print("=== Merging Visualization Examples (12 total) ===")
all_viz_examples = []
for asset in ASSETS:
    if asset in reports:
        examples = reports[asset].get("visualization_examples", [])
        all_viz_examples.extend(examples)
        print(f"  {asset}: {len(examples)} examples")

viz_path = os.path.join(REPO_ROOT, "results", "visualization_examples.json")
with open(viz_path, "w") as f:
    json.dump(all_viz_examples, f, indent=2)
print(f"\nSaved {len(all_viz_examples)} examples to: {viz_path}")

for ex in all_viz_examples:
    print(f"  {ex['asset']:10s} {ex['difficulty']:6s}: "
          f"shot_id={ex['shot_id']}, size={ex['gather_size']}, "
          f"label_frac={ex['label_fraction']:.2f}")

=== Merging Visualization Examples (12 total) ===
  brunswick: 3 examples
  halfmile: 3 examples
  lalor: 3 examples
  sudbury: 3 examples

Saved 12 examples to: /content/drive/MyDrive/seismic-first-break-picking/results/visualization_examples.json
  brunswick  easy  : shot_id=760, size=3346, label_frac=0.75
  brunswick  medium: shot_id=514, size=3115, label_frac=0.73
  brunswick  hard  : shot_id=144, size=2582, label_frac=0.80
  halfmile   easy  : shot_id=20301261, size=1604, label_frac=0.85
  halfmile   medium: shot_id=20181346, size=1604, label_frac=0.91
  halfmile   hard  : shot_id=20021527, size=1578, label_frac=0.97
  lalor      easy  : shot_id=910, size=2685, label_frac=0.59
  lalor      medium: shot_id=429, size=2685, label_frac=0.51
  lalor      hard  : shot_id=922, size=2685, label_frac=0.71
  sudbury    easy  : shot_id=243, size=1983, label_frac=0.15
  sudbury    medium: shot_id=866, size=1763, label_frac=0.06
  sudbury    hard  : shot_id=122, size=1510, label_frac=0.19


In [9]:
# ============================================================
# Cell 9 — EDA Summary & Recommendations (Step 1.8)
# ============================================================
print("=" * 80)
print("EDA SUMMARY DOCUMENT")
print("=" * 80)

summary = {
    "phase": "Phase 1 EDA",
    "status": "COMPLETE",
    "assets_analyzed": list(reports.keys()),
}

# Per-asset summary
per_asset = {}
for asset in ASSETS:
    if asset not in reports:
        continue
    r = reports[asset]
    meta = CFG["asset_meta"][asset]
    entry = {
        "total_traces": meta["total_traces"],
        "labeled_traces": meta["labeled_traces"],
        "labeled_fraction": meta["labeled_fraction"],
        "samp_rate_us": meta["samp_rate_us"],
        "samp_num": meta["samp_num"],
        "time_window_ms": meta["time_window_ms"],
        "n_shots": r.get("shot_gather_stats", {}).get("n_shots"),
        "median_gather_size": r.get("shot_gather_stats", {}).get("gather_size", {}).get("median"),
        "fb_time_median_ms": r.get("label_audit", {}).get("fb_time_ms", {}).get("median"),
        "fb_time_std_ms": r.get("label_audit", {}).get("fb_time_ms", {}).get("std"),
        "dead_fraction": r.get("amplitude_stats", {}).get("dead_fraction_estimate"),
        "clipped_fraction": r.get("amplitude_stats", {}).get("clipped_fraction_estimate"),
        "mispick_fraction": r.get("label_quality", {}).get("outlier_fraction"),
    }
    per_asset[asset] = entry
summary["per_asset"] = per_asset

# Cross-dataset issues
summary["critical_issues"] = [
    {
        "issue": "heterogeneous_sample_rates",
        "description": "Lalor uses 1000µs, others use 2000µs. Resampling required.",
        "affected_assets": ["lalor"],
    },
    {
        "issue": "heterogeneous_trace_lengths",
        "description": "751 (brunswick/halfmile), 1001 (sudbury), 1501 (lalor). Must unify.",
        "affected_assets": ASSETS,
    },
    {
        "issue": "extreme_label_imbalance",
        "description": "Sudbury has only 11% labeled traces vs 90% for Halfmile.",
        "affected_assets": ["sudbury", "lalor"],
    },
    {
        "issue": "different_coordinate_scales",
        "description": "COORD_SCALE varies: -10, 1, 100. Must normalize offsets.",
        "affected_assets": ASSETS,
    },
]

# Recommendations
summary["preprocessing_recommendations"] = [
    "Resample Lalor traces from 1ms to 2ms sampling to match other assets",
    "Crop/pad all traces to a common time window (1500ms recommended)",
    "Normalize coordinates per-asset before computing offsets for shot gather ordering",
    "Use shot-gather (2D) framing for primary model — trace-level framing as baseline",
    "Implement balanced sampling across assets weighted by label count",
    "Exclude unlabeled traces (SPARE1 <= 0) from training, keep for potential semi-supervised use",
    "Apply per-gather amplitude normalization (not global) to handle scale differences",
    "Use leave-one-site-out cross-validation (hardpicks benchmark protocol)",
]

summary["model_framing_decision"] = {
    "primary": "shot_gather_2d_segmentation",
    "baseline": "trace_level_regression",
    "reasoning": "2D shot gathers exploit spatial coherence of first-break curve. "
                 "This matches the hardpicks benchmark baseline (U-Net segmentation)."
}

summary["benchmark_reference"] = {
    "name": "hardpicks (mila-iqia)",
    "repo": "https://github.com/mila-iqia/hardpicks",
    "paper": "St-Charles et al., GEOPHYSICS 2023",
    "baseline_model": "U-Net semantic segmentation",
    "cross_validation": "leave-one-site-out (4 folds)",
}

# Print summary
for k, v in summary.items():
    if isinstance(v, dict):
        print(f"\n{k}:")
        for kk, vv in v.items():
            if isinstance(vv, dict):
                print(f"  {kk}:")
                for kkk, vvv in vv.items():
                    print(f"    {kkk}: {vvv}")
            else:
                print(f"  {kk}: {vv}")
    elif isinstance(v, list):
        print(f"\n{k}:")
        for item in v:
            if isinstance(item, dict):
                print(f"  - {item}")
            else:
                print(f"  - {item}")
    else:
        print(f"{k}: {v}")

EDA SUMMARY DOCUMENT
phase: Phase 1 EDA
status: COMPLETE

assets_analyzed:
  - brunswick
  - halfmile
  - lalor
  - sudbury

per_asset:
  brunswick:
    total_traces: 4496540
    labeled_traces: 3733221
    labeled_fraction: 0.8302
    samp_rate_us: 2000
    samp_num: 751
    time_window_ms: 1500
    n_shots: 1541
    median_gather_size: 2975.0
    fb_time_median_ms: 384.0
    fb_time_std_ms: 229.56444516376504
    dead_fraction: 0.0
    clipped_fraction: 0.0
    mispick_fraction: 0.003942002156944576
  halfmile:
    total_traces: 1099559
    labeled_traces: 993189
    labeled_fraction: 0.9033
    samp_rate_us: 2000
    samp_num: 751
    time_window_ms: 1500
    n_shots: 690
    median_gather_size: 1604.0
    fb_time_median_ms: 344.0
    fb_time_std_ms: 154.91804985078338
    dead_fraction: 0.01358
    clipped_fraction: 0.0
    mispick_fraction: 0.00470359369843503
  lalor:
    total_traces: 2424923
    labeled_traces: 1211857
    labeled_fraction: 0.4998
    samp_rate_us: 1000
    sam

In [10]:
# ============================================================
# Cell 10 — Save Combined Summary & Final Banner
# ============================================================
summary_path = os.path.join(EDA_JSON_DIR, "eda_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved: {summary_path}")

# List all generated combined plots
combined_plots = sorted(os.listdir(COMBINED_PLOT_DIR))
print(f"\nGenerated {len(combined_plots)} combined plots in {COMBINED_PLOT_DIR}:")
for p in combined_plots:
    sz = os.path.getsize(os.path.join(COMBINED_PLOT_DIR, p))
    print(f"  {p}  ({sz/1024:.0f} KB)")

# Verify all deliverables exist
print(f"\n--- Gate 2 Deliverable Check ---")
deliverables = [
    ("EDA Summary JSON", summary_path),
    ("Visualization Examples", viz_path),
]
for asset in ASSETS:
    deliverables.append(
        (f"{asset} EDA report",
         os.path.join(EDA_JSON_DIR, f"{asset}_eda_report.json")))

all_ok = True
for name, path in deliverables:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"  {status} {name}: {path}")
    if not exists:
        all_ok = False

print(f"\n{'=' * 60}")
if all_ok:
    print(f"PHASE 1 EDA COMPLETE — ALL DELIVERABLES PRESENT")
else:
    print(f"PHASE 1 EDA INCOMPLETE — MISSING DELIVERABLES")
print(f"{'=' * 60}")

Saved: /content/drive/MyDrive/seismic-first-break-picking/results/eda/eda_summary.json

Generated 2 combined plots in /content/drive/MyDrive/seismic-first-break-picking/results/eda_plots/combined:
  combined_fb_stats.png  (87 KB)
  combined_gather_sizes.png  (45 KB)

--- Gate 2 Deliverable Check ---
  ✓ EDA Summary JSON: /content/drive/MyDrive/seismic-first-break-picking/results/eda/eda_summary.json
  ✓ Visualization Examples: /content/drive/MyDrive/seismic-first-break-picking/results/visualization_examples.json
  ✓ brunswick EDA report: /content/drive/MyDrive/seismic-first-break-picking/results/eda/brunswick_eda_report.json
  ✓ halfmile EDA report: /content/drive/MyDrive/seismic-first-break-picking/results/eda/halfmile_eda_report.json
  ✓ lalor EDA report: /content/drive/MyDrive/seismic-first-break-picking/results/eda/lalor_eda_report.json
  ✓ sudbury EDA report: /content/drive/MyDrive/seismic-first-break-picking/results/eda/sudbury_eda_report.json

PHASE 1 EDA COMPLETE — ALL DELIVERA